In [3]:
# src/data_processing/streaming/kafka_producer.py
import json
from typing import Dict, Any, Optional
from kafka import KafkaProducer
from kafka.errors import KafkaError

import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parents[2]  # trỏ về thư mục project
sys.path.append(str(PROJECT_ROOT))

from src.utils.config import config
from src.utils.logger import LoggerMixin

In [4]:
class TrafficDataProducer(LoggerMixin):
    """
    Kafka Producer để gửi dữ liệu giao thông vào Kafka topics
    """
    
    def __init__(self, bootstrap_servers: Optional[str] = None):
        self.bootstrap_servers = bootstrap_servers or config.kafka.bootstrap_servers
        self.producer = None
        self._connect()
    
    def _connect(self):
        """Khởi tạo Kafka producer"""
        try:
            self.producer = KafkaProducer(
                bootstrap_servers=self.bootstrap_servers,
                value_serializer=lambda v: json.dumps(v).encode('utf-8'),
                key_serializer=lambda k: k.encode('utf-8') if k else None,
                acks='all',
                retries=3,
                max_in_flight_requests_per_connection=1,  # Ensure ordering
                compression_type="gzip"
            )
            self.logger.info(f"✅ Connected to Kafka: {self.bootstrap_servers}")
        except Exception as e:
            self.logger.error(f"❌ Failed to connect to Kafka: {e}")
            raise
    
    def send_raw_data(self, data: Dict[str, Any], key: Optional[str] = None):
        """
        Gửi dữ liệu thô vào topic traffic.raw
        
        Args:
            data: Dữ liệu cần gửi
            key: Message key (segment_id, job_id, etc.)
        """
        return self._send(config.kafka.raw_topic, data, key)
    
    def send_validated_data(self, data: Dict[str, Any], key: Optional[str] = None):
        """Gửi dữ liệu đã validate vào topic traffic.validated"""
        return self._send(config.kafka.validated_topic, data, key)
    
    def send_features(self, data: Dict[str, Any], key: Optional[str] = None):
        """Gửi features đã extract vào topic traffic.features"""
        return self._send(config.kafka.features_topic, data, key)
    
    def _send(self, topic: str, data: Dict[str, Any], key: Optional[str] = None):
        """
        Gửi message vào topic

        Args:
            topic (str): Tên topic
            data (Dict[str, Any]): Dữ liệu
            key (Optional[str], optional): Message Key. Defaults to None.
        """
        try:
            future = self.producer.send(topic, value=data, key=key)
            
            # Block để đảm bảo message đã được gửi
            record_metadata = future.get(timeout=10)
            
            self.logger.debug(
                f"Sent to {topic} - "
                f"partition: {record_metadata.partition}, "
                f"offset: {record_metadata.offset}"
            )
            return record_metadata
            
        except KafkaError as e:
            self.logger.error(f"❌ Kafka error when sending to {topic}: {e}")
            raise
        except Exception as e:
            self.logger.error(f"❌ Error when sending to {topic}: {e}")
            raise
    
    def flush(self):
        """Flush tất cả pending messages"""
        if self.producer:
            self.producer.flush()
            self.logger.debug("Flush all pending messages")

    def close(self):
        """Close producer"""
        if self.producer:
            self.producer.close()
            self.logger.info("Closed Kafka producer")

In [5]:
# class TrafficProducer:
#     def __init__(self, brokers: str):
#         try:
#             self.producer = KafkaProducer(
#                 bootstrap_servers=brokers,
#                 value_serializer=lambda v: json.dumps(v).encode("utf-8")
#             )
#         except NoBrokersAvailable:
#             raise RuntimeError(
#                 f"Kafka broker not available at {brokers}. "
#                 "Make sure Kafka is running."
#             )

#     def send(self, topic: str, data: Dict[str, Any]):
#         try:
#             self.producer.send(topic, value=data).get(timeout=5)
#         except KafkaError as e:
#             print("Kafka send failed:", e)

# producer = TrafficProducer("localhost:9092")

# producer.send(
#     "traffic.raw",
#     {"speed": 38.2, "confidence": 0.91}
# )

In [6]:
producer = TrafficDataProducer("localhost:9092")

data = {
    "segment_id": "SG_TEST_01",
    "speed": 38.2,
    "confidence": 0.91
}

producer.send_raw_data(data, key="SG_TEST_01")
producer.flush()
producer.close()

print("✅ Test producer done")

2025-12-17 10:39:58,534 - TrafficDataProducer - INFO - 2336633585.py:23 - ✅ Connected to Kafka: localhost:9092
2025-12-17 10:39:58,695 - TrafficDataProducer - INFO - 2336633585.py:85 - Closed Kafka producer
✅ Test producer done
